# BAI example

One BAI batch: data -> fixed-model predictions -> cube-balanced selection -> label -> inference.

Run from `examples/simulation/`, so that `dataset` and `model` import directly.

In [1]:
from model.judge import JudgeModel
from openppi import Mean, get_method, select_for_labeling
from dataset.judge_sim import make_judge_data

In [2]:
# 1. Separate features and labels from the data provider's DataFrame.
data = make_judge_data(n=2000, seed=0)
features = data.drop(columns='label')
judge_cols = [i for i, name in enumerate(features.columns) if name.startswith('judge')]
features = features.to_numpy()
y = data['label'].to_numpy()

In [3]:
# 2. Select the target and predict with a fixed model (any already trained model works).
target = Mean()
recipe = get_method('BAI')          # sampling: BalancedDesign(tau=.5); inference: BAI
model = JudgeModel(columns=judge_cols)
predictions = model.predict(features)

In [4]:
# 3. Same probabilities as ASI, but the cube method selects rows jointly so that the
#    weighted uncertainty of the batch matches the pool total. No label is read here.
#    With a signed error model e(x) ~ f(x) - y, pass BalancedDesign(balancing=e) and scores=|e|.
request = select_for_labeling(predictions, target=target, design=recipe.sampling(),
                              budget=400, random_state=42)
print('Rows to label:', len(request.indices))

Rows to label: 399


In [5]:
# 4. Human labeling. The simulation looks up y; in practice, export request.indices to annotators.
labels = y[request.indices]

In [6]:
# 5. Attach labels (-> BalancedActiveData) and run inference with the cube variance estimator.
record = request.with_labels(labels)
result = recipe.inference(record, target=target, alpha=.05)
print('Estimate:', result.estimate)
print('Confidence interval:', result.confidence_interval)
print('Diagnostics:', result.diagnostics)

Estimate: 0.5348817504488079
Confidence interval: (0.4864784787654113, 0.5832850221322046)
Diagnostics: {'audits': 399, 'sum_sampling_probabilities': 400.00000000000006, 'min_pi': 0.10624507259813676, 'max_pi': 0.24981901548168137, 'design_variance': 0.000485323847424537, 'population_variance': 0.00012456859683010875, 'relative_imbalance': [0.00209797789975082], 'variance_design': 'cube_plus_population'}
